# AdaPT sweep: 2 datasets x 3 models x 6 multipliers x (eval-only, retrained)

Writes one row per `(dataset, model, multiplier, retrained_epochs)` to `results/runs.csv` and a per-run confusion matrix to `results/confusion_matrices/<dataset>/<model>_<multiplier>_e<epochs>.npy`.

Models come from `chenyaofo/pytorch-cifar-models`, which has pretrained weights for CIFAR-10 and CIFAR-100. `wrap_with_adapt` swaps every `nn.Conv2d` in-place for `AdaPT_Conv2d`, so the approximate kernel runs in every conv layer.

Each `(dataset, model)` also gets one `fp32` reference row — the raw float model with no quantization or approximation. It is the fp32→int8 anchor for the architecture comparison (TransAxx already records the matching `fp32` rows for the ViTs); being a plain float eval, it has no retrained variant.

Re-running is safe: rows already in the CSV are skipped. To recompute everything from scratch (e.g. after changing the recipe or LUTs), set `FRESH = True` in the config cell.

In [1]:
import csv, gc, os, time, uuid
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import CIFAR10, CIFAR100, ImageFolder  # ImageFolder: bring-your-own-dataset example in make_loaders
from tqdm import tqdm

from adapt.approx_layers.axx_layers import AdaPT_Conv2d
from adapt.references.classification.train import train_one_epoch
from pytorch_quantization import nn as quant_nn, calib

torch.set_num_threads(40)
%env OMP_PLACES=cores
%env OMP_PROC_BIND=close
%env OMP_WAIT_POLICY=active

env: OMP_PLACES=cores
env: OMP_PROC_BIND=close
env: OMP_WAIT_POLICY=active


## LUT-level error metrics

A quick sanity table before the sweep: a pure-numpy diff of each multiplier's LUT against `mul8s_acc`. It lets you eyeball whether the headers under `adapt/cpu-kernels/axx_mults/` match what you expect.

In [2]:
import re
# absolute path to AdaPT's JIT kernel header dir (cwd is /work/notebooks now)
AXX_DIR = Path("/workspace/adapt/adapt/cpu-kernels/axx_mults")

def load_lut(name):
    text = (AXX_DIR / f"{name}.h").read_text()
    nums = re.findall(r"-?\d+", text.split("{", 1)[1])
    return np.array(nums, dtype=np.int16).reshape(256, 256)

def signed_view(lut):
    perm = np.r_[0:128, -128:0]
    order = np.argsort(perm)
    return lut[np.ix_(order, order)]

def lut_metrics(name, ref="mul8s_acc"):
    L = signed_view(load_lut(name)).astype(np.int32)
    R = signed_view(load_lut(ref)).astype(np.int32)
    err = L - R
    abs_err = np.abs(err).astype(np.float64)
    nonzero = R != 0
    rel = np.divide(abs_err, np.abs(R), out=np.zeros_like(abs_err), where=nonzero)
    return {
        "name": name,
        "ER":   float((err != 0).mean()),
        "MAE":  float(abs_err.mean()),
        "WCE":  int(np.abs(err).max()),
        "MSE":  float((err.astype(np.float64) ** 2).mean()),
        "MRED": float(rel[nonzero].mean()),
        "NMED": float(abs_err.mean() / (127 * 127)),
        "bias": float(err.mean()),
    }

_lut_names = [f"mul8s_fpga_ish_{i}" for i in range(1, 6)]
pd.DataFrame([lut_metrics(n) for n in _lut_names])

,name,ER,MAE,WCE,MSE,MRED,NMED,bias
0,mul8s_fpga_ish_1,0.334229,49.149414,714,19740.25,0.013288,0.003047,0.0
1,mul8s_fpga_ish_2,0.314453,19.818359,202,2196.25,0.008061,0.001229,0.0
2,mul8s_fpga_ish_3,0.288086,18.137695,170,2024.25,0.007219,0.001125,0.0
3,mul8s_fpga_ish_4,0.261719,17.075195,168,2040.25,0.006552,0.001059,0.0
4,mul8s_fpga_ish_5,0.226562,17.000000,168,2039.50,0.006434,0.001054,0.0


## Sweep config

**Models / datasets / parameters are set in `0_configure_sweeps.ipynb`** (edit there, Run All,
then re-run this notebook, since the cell below just reads its `sweep_config.json`). Only `FRESH`
lives here, because it deletes results. Results path is the `results/` volume mount, which
on the host maps to `docker/results/adapt/`.

In [3]:
import json as _json

# What to sweep comes from 0_configure_sweeps' manifest (EDIT THERE, not here,
# since this file is the single place models/datasets/params are defined). The literals
# below are only fallbacks for when the manifest hasn't been written yet.
_cfg_all = _json.loads(Path("sweep_config.json").read_text()) \
           if Path("sweep_config.json").exists() else {}
_cfg = _cfg_all.get("adapt", {})
DATASETS       = _cfg.get("datasets", ["cifar10", "cifar100"])
MODELS         = _cfg.get("models", ["resnet20", "resnet56", "vgg11_bn"])
RETRAIN_EPOCHS = _cfg.get("retrain_epochs", 3)
BATCH_SIZE     = _cfg.get("batch_size", 128)
CALIB_BATCHES  = _cfg.get("calib_batches", 2)
# Local user-provided models (docker/models/<name>.py + weights, mounted at ../models),
# registered in 0_configure_sweeps. Each runs only on its declared dataset.
LOCAL_MODELS   = {n: s for n, s in _cfg_all.get("local_models", {}).items()
                  if s.get("framework") == "adapt"}

# Multipliers come from 0_build_multipliers' manifest (define once, sweep everywhere).
# mul8s_exact is the generator's exact twin of the mul8s_acc baseline, so we keep acc
# and drop the duplicate. Falls back to the fixed list if the manifest is not built yet.
_manifest = Path("../luts/multipliers.json")
if _manifest.exists():
    _built = _json.loads(_manifest.read_text())["multipliers"]
    MULTS = [m for m in _built if m != "mul8s_exact"] + ["mul8s_acc"]
else:
    MULTS = [f"mul8s_fpga_ish_{i}" for i in range(1, 6)] + ["mul8s_acc"]

# FRESH = False -> resume: rows already in runs.csv are skipped, so you can
#                  interrupt and re-run safely. NOTE: it only checks whether a
#                  row exists, not whether it is up to date -- if you change the
#                  recipe / LUTs / calibration it will NOT recompute existing rows.
# FRESH = True  -> wipe runs.csv + confusion_matrices/ (and legacy runs/) first and
#                  recompute the whole matrix from scratch.
FRESH = False

# Jupyter roots at /work, and this notebook lives in /work/notebooks, so results go to
# /work/results/adapt (compose mounts ./results -> /work/results).
RESULTS_DIR = Path("../results/adapt")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_DIR / "runs.csv"
COLS = ["run_id", "timestamp", "dataset", "model", "multiplier",
        "retrained_epochs", "top1", "top5", "n_samples", "runtime_s", "notes"]

if FRESH:
    import shutil
    CSV_PATH.unlink(missing_ok=True)
    for _d in (RESULTS_DIR / "confusion_matrices", RESULTS_DIR / "runs"):
        shutil.rmtree(_d, ignore_errors=True)
    print("FRESH=True -> cleared runs.csv, confusion_matrices/ and legacy runs/")

print(f"Matrix size: {len(DATASETS)} x {len(MODELS)} x {len(MULTS)} x 2 = "
      f"{len(DATASETS)*len(MODELS)*len(MULTS)*2} rows")
if LOCAL_MODELS:
    print("+ local models: " + ", ".join(f"{n} (on {s['dataset']} only)"
                                          for n, s in LOCAL_MODELS.items()))

Matrix size: 2 x 3 x 9 x 2 = 108 rows


## Data loaders

A small `DATASETS` registry maps each dataset name to its normalisation stats, class count, and a `build(root, transform)` that returns the `(train, test)` datasets. `make_loaders` then just looks the name up. The CIFAR entries use the standard per-channel normalisation `chenyaofo/pytorch-cifar-models` was trained with. Adding a dataset (a torchvision one, or your own `ImageFolder`) is one entry here plus allowing the name in `0_configure_sweeps`' validator, as shown by the commented `myset` example. Calibration set is every 10th sample of train (10% subset, same convention as the upstream notebook).

In [4]:
# ---- Dataset registry -------------------------------------------------------
# One entry per dataset: normalisation stats, class count, and a build(root, transform)
# that returns the (train, test) torch Datasets. To ADD a dataset, add an entry here and
# allow its name in 0_configure_sweeps' validator -- nothing else in this notebook needs
# to change. Anything torchvision can build, or your own ImageFolder, drops in this way
# (see the commented "myset" example). Data lives under /workspace/datasets, bind-mounted
# from docker/datasets, so it persists across rebuilds and can be staged offline.
def _torchvision_cifar(cls):
    """Factory for the CIFAR-family API: cls(root, train=bool, download, transform)."""
    def build(root, transform):
        train = cls(root=root, train=True,  download=True, transform=transform)
        test  = cls(root=root, train=False, download=True, transform=transform)
        return train, test
    return build

DATASETS = {
    "cifar10":  {"mean": (0.4914, 0.4822, 0.4465), "std": (0.2471, 0.2435, 0.2616),
                 "n_classes": 10,  "build": _torchvision_cifar(CIFAR10)},
    "cifar100": {"mean": (0.5071, 0.4867, 0.4408), "std": (0.2675, 0.2565, 0.2761),
                 "n_classes": 100, "build": _torchvision_cifar(CIFAR100)},

    # ---- Example: bring-your-own dataset (uncomment + allow "myset" in the validator
    # of 0_configure_sweeps to use). Put the images under
    #   docker/datasets/myset_data/{train,test}/<class_name>/*.jpg
    # and ImageFolder reads the class labels straight from the folder names. Use real
    # per-channel mean/std for your data; (0.5, 0.5, 0.5) is just a placeholder.
    # "myset": {
    #     "mean": (0.5, 0.5, 0.5), "std": (0.5, 0.5, 0.5), "n_classes": 10,
    #     "build": lambda root, tf: (ImageFolder(f"{root}/train", transform=tf),
    #                                ImageFolder(f"{root}/test",  transform=tf)),
    # },
}

def make_loaders(dataset, batch_size=BATCH_SIZE):
    info = DATASETS[dataset]
    tf = T.Compose([T.ToTensor(), T.Normalize(info["mean"], info["std"])])
    # absolute path (cwd is /work/notebooks now); bind-mounted from docker/datasets.
    root = f"/workspace/datasets/{dataset}_data"
    train, test = info["build"](root, tf)
    val = DataLoader(test, batch_size=batch_size, num_workers=0,
                     drop_last=True, pin_memory=False)
    evens = list(range(0, len(train), 10))
    cal = DataLoader(torch.utils.data.Subset(train, evens),
                     batch_size=batch_size, shuffle=False,
                     num_workers=0, drop_last=True, pin_memory=False)
    return val, cal

## Model builder + AdaPT wrap

`build_model` pulls pretrained weights from the chenyaofo hub. `wrap_with_adapt` walks the module tree and swaps every `nn.Conv2d` for an `AdaPT_Conv2d` of the same shape, copying weights/bias across. Each new `AdaPT_Conv2d` JIT-compiles its C++ kernel on first instantiation per `axx_mult`, which is the slow first iteration of the sweep.

In [5]:
def build_model(name, dataset):
    hub_id = f"{dataset}_{name}"
    return torch.hub.load("chenyaofo/pytorch-cifar-models", hub_id, pretrained=True)

MODELS_DIR = Path("../models")   # docker/models on the host, /work/models in-container

def build_local_model(name):
    """Import docker/models/<name>.py and call its build_model(). The .py owns
    everything model-specific (architecture, weight loading, output selection),
    so this notebook never needs editing for a new model (see ../models/README.md)."""
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, (MODELS_DIR / f"{name}.py").resolve())
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.build_model()

def wrap_with_adapt(model, axx_mult):
    """Recursively replace nn.Conv2d -> AdaPT_Conv2d, copying weights/bias."""
    for name, module in model.named_children():
        if isinstance(module, nn.Conv2d) and not isinstance(module, AdaPT_Conv2d):
            new = AdaPT_Conv2d(
                module.in_channels, module.out_channels, module.kernel_size,
                stride=module.stride, padding=module.padding,
                dilation=module.dilation, groups=module.groups,
                bias=module.bias is not None, padding_mode=module.padding_mode,
                axx_mult=axx_mult,
            )
            new.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new.bias.data.copy_(module.bias.data)
            setattr(model, name, new)
        else:
            wrap_with_adapt(module, axx_mult)
    return model

## Calibration, evaluation, finetune helpers

In [6]:
def collect_stats(model, loader, num_batches):
    for _, m in model.named_modules():
        if isinstance(m, quant_nn.TensorQuantizer):
            if m._calibrator is not None:
                m.disable_quant(); m.enable_calib()
            else:
                m.disable()
    with torch.no_grad():
        for i, (img, _) in enumerate(loader):
            model(img.cpu())
            if i >= num_batches:
                break
    for _, m in model.named_modules():
        if isinstance(m, quant_nn.TensorQuantizer):
            if m._calibrator is not None:
                m.enable_quant(); m.disable_calib()
            else:
                m.enable()

def compute_amax(model, **kwargs):
    for _, m in model.named_modules():
        if isinstance(m, quant_nn.TensorQuantizer):
            if m._calibrator is not None:
                if isinstance(m._calibrator, calib.MaxCalibrator):
                    m.load_calib_amax()
                else:
                    m.load_calib_amax(**kwargs)
    model.cpu()

def calibrate(model, calib_loader):
    with torch.no_grad():
        collect_stats(model, calib_loader, num_batches=CALIB_BATCHES)
        compute_amax(model, method="percentile", percentile=99.99)

def evaluate(model, loader, n_classes):
    model.eval()
    correct = top5 = total = 0
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    t0 = time.time()
    with torch.no_grad():
        for img, lbl in tqdm(loader, total=len(loader), leave=False):
            out = model(img)
            _, p5 = out.topk(5, dim=1)
            p1 = p5[:, 0]
            total += lbl.size(0)
            correct += (p1 == lbl).sum().item()
            top5 += (p5 == lbl.unsqueeze(1)).any(dim=1).sum().item()
            for t, p in zip(lbl.numpy(), p1.numpy()):
                cm[t, p] += 1
    dt = time.time() - t0
    return 100 * correct / total, 100 * top5 / total, total, dt, cm

def finetune(model, loader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)
    for e in range(epochs):
        train_one_epoch(model, criterion, optimizer, loader, "cpu", e, 50)

In [7]:
def existing_keys():
    if not CSV_PATH.exists():
        return set()
    df = pd.read_csv(CSV_PATH)
    return {tuple(r) for r in df[["dataset", "model", "multiplier", "retrained_epochs"]]
                                  .itertuples(index=False, name=None)}

def append_row(row):
    new_file = not CSV_PATH.exists()
    with CSV_PATH.open("a", newline="") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(COLS)
        w.writerow([row.get(c, "") for c in COLS])

def cm_path(dataset, model_name, mult, retrained_epochs):
    """Human-readable, dataset-foldered confusion-matrix path, e.g.
    results/confusion_matrices/cifar100/resnet20_fpga_ish_1_e0.npy."""
    short = mult.replace("mul8s_", "")
    d = RESULTS_DIR / "confusion_matrices" / dataset
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{model_name}_{short}_e{retrained_epochs}.npy"

def record(dataset, model_name, mult, retrained_epochs, top1, top5, n, dt, cm):
    rid = uuid.uuid4().hex[:12]
    np.save(cm_path(dataset, model_name, mult, retrained_epochs), cm)
    append_row({
        "run_id": rid,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "dataset": dataset, "model": model_name, "multiplier": mult,
        "retrained_epochs": retrained_epochs,
        "top1": f"{top1:.4f}", "top5": f"{top5:.4f}",
        "n_samples": n, "runtime_s": f"{dt:.1f}", "notes": "",
    })
    return rid

## Sweep

Outer loop: dataset; middle: model; inner: multiplier. For each cell:

1. Build a fresh chenyaofo model, wrap with this multiplier's AdaPT_Conv2d, calibrate.
2. Eval-only row → append (`retrained_epochs=0`) + save confusion matrix.
3. Finetune `RETRAIN_EPOCHS` epochs on the calib subset, re-evaluate → append + save.

Before the multiplier loop, each model also gets one `fp32` row — the raw float model evaluated with no AdaPT wrap and no quantization (the fp32→int8 anchor for the architecture comparison; no retrained variant).

Rows that are already present in `results/runs.csv` (matched on `dataset`, `model`, `multiplier`, `retrained_epochs`) are skipped, so you can interrupt and resume.

In [8]:
done = existing_keys()
print(f"{len(done)} rows already in CSV; will skip those.\n")

for dataset in DATASETS:
    val, cal = make_loaders(dataset)
    n_cls = DATASETS[dataset]["n_classes"]
    # local models join the matrix on their declared dataset only
    local_here = [n for n, s in LOCAL_MODELS.items() if s["dataset"] == dataset]
    for model_name in MODELS + local_here:
        # FP32 reference: the raw float model, no AdaPT wrap and no quantization.
        # This is the fp32->int8 anchor for the architecture comparison (mirrors the ViT fp32 rows TransAxx
        # already has). It is a plain float eval, not an approximate-multiplier run,
        # so it sits outside the multiplier loop and has no retrained variant.
        if (dataset, model_name, "fp32", 0) not in done:
            print(f"\n=== {dataset} / {model_name} / fp32 (float reference) ===")
            fb = (build_local_model(model_name) if model_name in local_here
                  else build_model(model_name, dataset)).eval()
            top1, top5, n, dt, cm = evaluate(fb, val, n_cls)
            rid = record(dataset, model_name, "fp32", 0, top1, top5, n, dt, cm)
            print(f"  fp32           top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")
            del fb
            gc.collect()

        for mult in MULTS:
            need_eval     = (dataset, model_name, mult, 0) not in done
            need_retrain  = RETRAIN_EPOCHS > 0 and \
                            (dataset, model_name, mult, RETRAIN_EPOCHS) not in done
            if not (need_eval or need_retrain):
                print(f"[skip] {dataset}/{model_name}/{mult} — both rows present")
                continue

            print(f"\n=== {dataset} / {model_name} / {mult} ===")
            base = (build_local_model(model_name) if model_name in local_here
                    else build_model(model_name, dataset))
            m = wrap_with_adapt(base, mult).eval()
            calibrate(m, cal)

            if need_eval:
                top1, top5, n, dt, cm = evaluate(m, val, n_cls)
                rid = record(dataset, model_name, mult, 0, top1, top5, n, dt, cm)
                print(f"  eval-only      top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")

            if need_retrain:
                finetune(m, cal, RETRAIN_EPOCHS)
                top1, top5, n, dt, cm = evaluate(m, val, n_cls)
                rid = record(dataset, model_name, mult, RETRAIN_EPOCHS,
                             top1, top5, n, dt, cm)
                print(f"  retrained({RETRAIN_EPOCHS}ep) top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")

            del m, base
            gc.collect()

from IPython.display import FileLink, display

print("Done.")
print(f"Results saved under: {RESULTS_DIR.resolve()}\n")

# Live inventory of what the results folder holds right now.
n_rows = sum(1 for _ in open(CSV_PATH)) - 1
n_cm = len(list((RESULTS_DIR / "confusion_matrices").glob("*/*.npy")))
print(f"  runs.csv                                {n_rows} rows -- one per (dataset, model, multiplier, retrained/not)")
print(f"  confusion_matrices/<dataset>/<..>.npy   {n_cm} files  -- per-run confusion matrix (loaded by 3_multiplier_deepdive.ipynb)")

# Clickable link to the CSV, openable straight from this notebook.
print("\nOpen the results CSV:")
display(FileLink(str(CSV_PATH)))
pd.read_csv(CSV_PATH).tail(15)

123 rows already in CSV; will skip those.

Files already downloaded and verified
Files already downloaded and verified

=== cifar10 / resnet20 / mul8s_1kva ===


Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py310_cpu/PyInit_conv2d_mul8s_1kva...
Emitting ninja build file /root/.cache/torch_extensions/py310_cpu/PyInit_conv2d_mul8s_1kva/build.ninja...
Building extension module PyInit_conv2d_mul8s_1kva...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/2] c++ -MMD -MF axx_conv2d.o.d -DTORCH_EXTENSION_NAME=PyInit_conv2d_mul8s_1kva -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.10/dist-packages/torch/include -isystem /usr/local/lib/python3.10/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.10/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.10/dist-packages/torch/include/THC -isystem /usr/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++14 -DAXX_MULT=mul8s_1kva -march=native -fopenmp -O3 -c /workspace/adapt/adapt/cpu-kernels/axx_conv2d.cpp -o axx_conv2d.o 
[2/2] c++ axx_conv2d.o -shared -lgomp -L/usr/local/lib/python3.10/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o PyInit_conv2d_mul8s_1kva.so


Loading extension module PyInit_conv2d_mul8s_1kva...
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kva, skipping build step...
Loading extension module PyInit_conv2d_mul8s_1kva...
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kva, skipping build step...
Loading extension module PyInit_conv2d_mul8s_1kva...
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kva, skipping build step...
Loading extension module PyInit_conv2d_mul8s_1kva...
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kva, skipping build step...
Loading extension module PyInit_conv2d_mul8s_1kva...
Using /

  eval-only      top1=91.8470  top5=99.8197  (24.3s)  run_id=954d98c3ae28
Epoch: [0]  [ 0/39]  eta: 0:00:21  lr: 0.0001  img/s: 239.88827187082777  loss: 0.1729 (0.1729)  acc1: 94.5312 (94.5312)  acc5: 100.0000 (100.0000)  time: 0.5406  data: 0.0070
Epoch: [0] Total time: 0:00:18
Epoch: [1]  [ 0/39]  eta: 0:00:19  lr: 0.0001  img/s: 260.9932339343617  loss: 0.1679 (0.1679)  acc1: 95.3125 (95.3125)  acc5: 100.0000 (100.0000)  time: 0.4972  data: 0.0067
Epoch: [1] Total time: 0:00:18
Epoch: [2]  [ 0/39]  eta: 0:00:19  lr: 0.0001  img/s: 265.70191470155044  loss: 0.1605 (0.1605)  acc1: 95.3125 (95.3125)  acc5: 100.0000 (100.0000)  time: 0.4923  data: 0.0105
Epoch: [2] Total time: 0:00:18


...

Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kvb, skipping build step...
Loading extension module PyInit_conv2d_mul8s_1kvb...
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kvb, skipping build step...
Loading extension module PyInit_conv2d_mul8s_1kvb...
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kvb, skipping build step...
Loading extension module PyInit_conv2d_mul8s_1kvb...
Using /root/.cache/torch_extensions/py310_cpu as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_1kvb, skipping build step...
Loading extension module PyInit

  eval-only      top1=63.4215  top5=83.6639  (70.7s)  run_id=6aafa7d8b1bc
Epoch: [0]  [ 0/39]  eta: 0:00:43  lr: 0.0001  img/s: 114.87636737351521  loss: 0.0406 (0.0406)  acc1: 98.4375 (98.4375)  acc5: 100.0000 (100.0000)  time: 1.1234  data: 0.0092
Epoch: [0] Total time: 0:00:43
Epoch: [1]  [ 0/39]  eta: 0:00:43  lr: 0.0001  img/s: 115.16898021770669  loss: 0.0525 (0.0525)  acc1: 98.4375 (98.4375)  acc5: 100.0000 (100.0000)  time: 1.1186  data: 0.0072
Epoch: [1] Total time: 0:00:43
Epoch: [2]  [ 0/39]  eta: 0:00:43  lr: 0.0001  img/s: 116.72251170712607  loss: 0.0574 (0.0574)  acc1: 99.2188 (99.2188)  acc5: 100.0000 (100.0000)  time: 1.1071  data: 0.0105
Epoch: [2] Total time: 0:00:43


  retrained(3ep) top1=67.7885  top5=86.5284  (70.4s)  run_id=b5c7739d8ca9
[skip] cifar100/vgg11_bn/mul8s_1l2h — both rows present
[skip] cifar100/vgg11_bn/mul8s_fpga_ish_1 — both rows present
[skip] cifar100/vgg11_bn/mul8s_fpga_ish_2 — both rows present
[skip] cifar100/vgg11_bn/mul8s_fpga_ish_3 — both rows present
[skip] cifar100/vgg11_bn/mul8s_fpga_ish_4 — both rows present
[skip] cifar100/vgg11_bn/mul8s_fpga_ish_5 — both rows present
[skip] cifar100/vgg11_bn/mul8s_acc — both rows present
Done.
Results saved under: /work/results/adapt

  runs.csv                                147 rows -- one per (dataset, model, multiplier, retrained/not)
  confusion_matrices/<dataset>/<..>.npy   147 files  -- per-run confusion matrix (loaded by 3_multiplier_deepdive.ipynb)

Open the results CSV:


/work/results/adapt/runs.csv

,run_id,timestamp,dataset,model,multiplier,retrained_epochs,top1,top5,n_samples,runtime_s,notes
132,b1befb7fe19f,2026-06-26T18:10:12,cifar10,vgg11_bn,mul8s_1kva,3,91.3462,99.5393,9984,68.5,NaN
133,ffa223a00f80,2026-06-26T18:11:22,cifar10,vgg11_bn,mul8s_1kvb,0,89.7135,99.3790,9984,68.5,NaN
134,1c4280ce550a,2026-06-26T18:14:37,cifar10,vgg11_bn,mul8s_1kvb,3,91.3161,99.5693,9984,68.3,NaN
135,fd761be3deb7,2026-06-26T18:15:04,cifar100,resnet20,mul8s_1kva,0,67.7985,90.7652,9984,24.9,NaN
136,5bccb132fef8,2026-06-26T18:16:26,cifar100,resnet20,mul8s_1kva,3,62.7804,86.9792,9984,24.9,NaN
137,bcfae1b713db,2026-06-26T18:16:53,cifar100,resnet20,mul8s_1kvb,0,61.2881,86.6587,9984,25.1,NaN
138,4a2e3865e454,2026-06-26T18:18:15,cifar100,resnet20,mul8s_1kvb,3,62.9307,86.8389,9984,24.8,NaN
139,ee25f8d0f738,2026-06-26T18:19:31,cifar100,resnet56,mul8s_1kva,0,71.8950,91.7869,9984,74.2,NaN
140,c2f2c497ff82,2026-06-26T18:23:32,cifar100,resnet56,mul8s_1kva,3,67.6883,87.9708,9984,74.4,NaN
141,874dff6719ff,2026-06-26T18:24:49,cifar100,resnet56,mul8s_1kvb,0,66.0156,88.2913,9984,74.7,NaN
